# In-class Activity 14: Sentiment Analysis on Real Reviews

## Part 1: Load and Explore the Data

### Step 1: Load the data (default settings)

In [17]:
import pandas as pd

DATA_PATH = "data/amazon_reviews_us_Digital_Video_Games_v1_00.tsv.zip"

df = pd.read_csv(DATA_PATH, sep="\t")
print("Default settings:", df.shape)

Default settings: (144724, 15)


**How many rows and columns did pandas load?**
With default settings pandas loaded **144,724** rows and **15** columns.

### Step 2: Load it again, correctly

In [18]:
df = pd.read_csv(DATA_PATH, sep="\t", quoting=3)  # 3 = treat quotes as plain text
print("quoting=3:", df.shape)

quoting=3: (145431, 15)


**How many rows does each version load? Did pandas warn you about the difference?**
Default loaded **144,724** rows and quoting=3 loaded **145,431** rows, so the default version silently lost **707** rows. Pandas did not warn me. A stray quote mark makes pandas think a field keeps going, so it swallows the following lines into one giant "review" with no error.

**What does that tell you about checking your data?**
"It ran without errors" doesn't mean it loaded correctly. You have to sanity check row counts against what you expect (the dataset should have about 145,000 reviews) and spot check the actual rows before trusting any analysis built on top of it.

### Step 3: Read some actual reviews

In [19]:
peek = df["review_body"].sample(5, random_state=2134)
for text in peek:
    print(text, "\n")

The &#34;AI&#34; is wrong by cheating, many bugs and control system cheating is not reliable<br />the PC version is penalized 

Is fast secure and very easy you only payed and in less than a minute you can use the code thanks 

This is exactly what it says it is, and I was able to email it as a gift!  Perfect.  Thanks! 

Since there is no way to return downloaded games or even remark on my problem, my only other options is via a review.<br /><br />I have a steam account and am logged in.  I tried installing the game, and it gave me a generic &#34;Trouble Communicating With Gateway&#34; message.  Googling the message returns one web page in German.  Retried three times with the same result.<br /><br />So...  Looks like I'm screwed. 

I bought this crapware online but was not able to install it on my tablet nor my android phone. The amazon android market app \\"seems\\" to crash when installing free or promotional apps. Amazon customer service is non existent. They dont care once they ha

**What do you notice that looks weird in these reviews?**
The text has leftover web formatting mixed in. There are `<br />` tags (sometimes doubled as `<br /><br />`) where line breaks should be, HTML entities like `&#34;` instead of actual quote marks (`&#34;AI&#34;`), and escaped quotes with backslashes like `\\"seems\\"`. There are also normal typos and grammar slips ("payed", "dont", "my only other options is"). Also, several of these aren't really about a game. They're about buying a code, sending it as a gift, or the Amazon app store crashing, so the "review" is about the purchase experience, not gameplay.

**Where do you think it came from?**
The reviews were typed into Amazon's website and stored as HTML, so line breaks and quote marks got saved as raw markup (`<br />`, `&#34;`) instead of plain text. The backslashes look like the quotes were escaped when the data was exported to a text file and never cleaned up.

### Step 4: Look at the star ratings

In [20]:
df["star_rating"].value_counts(normalize=True).sort_index()

star_rating
1    0.171697
2    0.053283
3    0.079962
4    0.140314
5    0.554744
Name: proportion, dtype: float64

**How are the ratings distributed? Why might that matter when you measure a model's accuracy?**
The ratings are heavily skewed toward 5 stars (**55.5%** are 5-star), but it's not a smooth slope. 1-star is actually the second most common rating at **17.2%**, followed by 4-star at **14.0%**, while 3-star (**8.0%**) and 2-star (**5.3%**) are the rarest. So people mostly either love it or hate it, and lukewarm reviews are uncommon. That matters because accuracy can look great just by guessing the majority class. A model that called everything positive would score around **70%** (4+5 stars = 69.5%) on a random sample without understanding anything, and the in-between ratings (2 and 3 stars), where a model is most likely to struggle, barely count toward the score.

### Step 5: Look at the products

In [21]:
df["product_title"].value_counts().head(10)

product_title
Playstation Network Card                                          13642
Xbox Live Subscription                                             7307
Playstation Plus Subscription                                      4712
Xbox Live Gift Card                                                3438
SimCity - Limited Edition                                          3421
Xbox 360 Live Points Card                                          3036
Battlefield 4                                                      1453
Final Fantasy XIV: A Realm Reborn                                  1273
Battlefield 3                                                      1160
Command and Conquer The Ultimate Collection [Online Game Code]      875
Name: count, dtype: int64

**What do you notice about the most-reviewed "games"?**
Most of the top products aren't games at all. 5 of the top 6 are payment or subscription products: **Playstation Network Card** (13,642), **Xbox Live Subscription** (7,307), **Playstation Plus Subscription** (4,712), **Xbox Live Gift Card** (3,438) and **Xbox 360 Live Points Card** (3,036). Together that's about 32,000 reviews, roughly 22% of the whole dataset. So "video game reviews" is largely reviews of gift cards and memberships, which people judge on price, how fast the code arrives and account issues rather than gameplay. The actual games that do show up (SimCity, Battlefield 3 and 4, Final Fantasy XIV) are big online titles, where reviews are often about servers, connection problems and launch bugs rather than the game itself.

### Step 6: Clean the text

In [22]:
print("Reviews with <br />:", df["review_body"].str.contains("<br />").sum())
df = df.dropna(subset=["review_body"])
df["review_body"] = df["review_body"].str.replace("<br />", " ")
print("After cleaning:", df["review_body"].str.contains("<br />").sum(), "\n")

for text in df.loc[df.index.intersection(peek.index), "review_body"]:
    print(text, "\n")

Reviews with <br />: 25699
After cleaning: 0 

Is fast secure and very easy you only payed and in less than a minute you can use the code thanks 

The &#34;AI&#34; is wrong by cheating, many bugs and control system cheating is not reliable the PC version is penalized 

Since there is no way to return downloaded games or even remark on my problem, my only other options is via a review.  I have a steam account and am logged in.  I tried installing the game, and it gave me a generic &#34;Trouble Communicating With Gateway&#34; message.  Googling the message returns one web page in German.  Retried three times with the same result.  So...  Looks like I'm screwed. 

This is exactly what it says it is, and I was able to email it as a gift!  Perfect.  Thanks! 

I bought this crapware online but was not able to install it on my tablet nor my android phone. The amazon android market app \\"seems\\" to crash when installing free or promotional apps. Amazon customer service is non existent. They 

**How many reviews had line breaks before cleaning, and after?**
**25,699** reviews had `<br />` before cleaning and **0** after.

**What still looks weird in the five reviews? Do you think it would change whether a model reads a review as positive or negative?**
The `<br />` tags are gone, but the HTML entities are still there (`&#34;AI&#34;`, `&#34;Trouble Communicating With Gateway&#34;`), and so are the escaped quotes with backslashes (`\\"seems\\"`, `\\"free\\"`). There are also double spaces where `<br /><br />` used to be, plus the same typos ("payed", "dont"). I don't think it would change the overall positive/negative call much, because the words that carry the sentiment ("crapware", "Looks like I'm screwed", "Perfect. Thanks!") are still intact. The one thing it could affect is sarcasm: the quotes around "seems" and "free" signal that the writer is being sarcastic, and when

### Step 7: Take a sample

In [23]:
sample = df.groupby("star_rating").sample(n=100, random_state=42)
complaints = df[df["star_rating"] == 1].sample(n=20, random_state=42)
print(len(sample), len(complaints))

500 20


**Why take the same number of each star rating instead of 500 random reviews?**
Because of the skew from Step 4. A random 500 would be about 277 five-star reviews (55.5%) but only about 26 two-star (5.3%) and 40 three-star (8.0%) reviews. That's too few to tell how the model handles negative or mixed reviews, and the overall accuracy would be inflated by the easy majority class. Taking 100 of each rating (a stratified sample) gives every rating an equal say, so we can see where the model actually struggles.

## Part 2: Run a Sentiment Model

### Step 8: Try one review

In [24]:
from transformers import pipeline

classifier = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
)
print(sample["star_rating"].iloc[0], sample["review_body"].iloc[0][:200])
classifier(sample["review_body"].iloc[0])

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

1 Terrible, confusing interface. Looks like a cartoon. Didn't really like the long, drawn out stories in my player. Game play and graphics weren't that much improved $30 extra we had to pay (Now we get 


[{'label': 'NEGATIVE', 'score': 0.9998071789741516}]

**What are the keys in the result, and what does the score mean?**
The result is a list with one dict that has two keys: `label` (POSITIVE or NEGATIVE) and `score`. The score is the model's confidence (a probability from 0.5 to 1) in the label it picked, not a measure of how positive or negative the review is.

**Does the model agree with the customer's star rating?**
Yes. The review is **1 star** and the model said **NEGATIVE** with a score of **0.9998**, so it agrees, and it's almost completely sure. That makes sense, since the review opens with "Terrible, confusing interface" and complains about paying $30 extra.

### Step 9: Classify all 500 reviews in one call

In [25]:
results = classifier(sample["review_body"].tolist(), truncation=True)
sample["label"] = [r["label"] for r in results]
sample["score"] = [r["score"] for r in results]
sample[["star_rating", "label", "score"]].head(10)

,star_rating,label,score
40836,1,NEGATIVE,0.999807
85497,1,NEGATIVE,0.990978
124434,1,NEGATIVE,0.998919
118528,1,NEGATIVE,0.999704
13480,1,NEGATIVE,0.999466
47179,1,NEGATIVE,0.999234
130193,1,NEGATIVE,0.999613
126902,1,NEGATIVE,0.999815
58985,1,NEGATIVE,0.999799
22608,1,NEGATIVE,0.999736


**About how long did 500 reviews take? At that rate, how long would all 145,000 take?**
500 reviews took about **6.4** seconds. 145,000 / 500 = 290, so all of them would take about 290 × 6.4 seconds ≈ **1,870 seconds, or about 31 minutes** on my laptop. That's much faster than "hours," but it's still too slow to rerun every time you tweak something, which is why sampling makes sense.

**What does `truncation=True` protect against?**
The model can only read 512 tokens at a time. Without truncation, a very long review (some are 5,000+ words) would make the model crash with an input-too-long error. With it, the review gets cut off at 512 tokens, so it runs, but the model only judges the beginning of long reviews.

## Part 3: Is the Model Any Good?

### Step 10: Create the answer key and score the model

In [26]:
def stars_to_sentiment(stars):
    """Convert a star rating into the sentiment the customer expressed."""
    if stars <= 2:
        return "NEGATIVE"
    if stars >= 4:
        return "POSITIVE"
    return "NEUTRAL"


sample["truth"] = sample["star_rating"].apply(stars_to_sentiment)
graded = sample[sample["truth"] != "NEUTRAL"]
print("Accuracy:", (graded["label"] == graded["truth"]).mean())

Accuracy: 0.8775


**Why do we leave out the 3-star reviews here?**
3 stars is neutral/mixed, and the model can only answer POSITIVE or NEGATIVE, so there's no "right" answer to grade it against. Including them would penalize the model for a category it can't output.

**What's the accuracy? Would you trust this model in a product?**
Accuracy is **87.75%** on the 400 graded reviews (351 right, 49 wrong). Because the sample is balanced (200 negative, 200 positive), always guessing one label would only get 50%, so this is genuinely decent for a model that was never trained on game reviews (it was trained on movie review sentences). But I wouldn't trust it alone in a product. Roughly **1 in 8** reviews is wrong, it has no neutral option, and it gives very high confidence scores (0.99+ on the reviews I looked at), so you can't easily tell when it's guessing. I'd use it for rough trends across many reviews, not for judging individual reviews.

### Step 11: Break it down by star rating

In [27]:
pd.crosstab(sample["star_rating"], sample["label"])

label,NEGATIVE,POSITIVE
star_rating,,
1,96,4
2,95,5
3,66,34
4,23,77
5,17,83


**Which star rating does the model get wrong most often?**
**4-star** reviews are wrong most often (23 of 100 called NEGATIVE), followed by **5-star** (17 of 100). The negative side is much more accurate: only 4 one-star and 5 two-star reviews were called POSITIVE. So 40 of the model's 49 mistakes are positive reviews it read as negative. The model leans negative. 4-star reviews are the "good, but..." ratings where customers praise the game and then list complaints ("fun but too short"), and the model latches onto the complaint. Even 5-star reviews often mention a problem they had or describe the game's content in negative-sounding words, which can trip it up.

**The model only knows POSITIVE and NEGATIVE. What did it do with the 3-star reviews?**
It forced every 3-star review into one side: **34** POSITIVE and **66** NEGATIVE. It leaned negative, which fits the pattern above. Mixed reviews usually spell out what went wrong, and the model weighs complaints more heavily than praise. Either way it's hiding the fact that these customers were lukewarm.

### Step 12: Read the mistakes

In [28]:
mistakes = graded[graded["label"] != graded["truth"]]
for review in mistakes.head(5).itertuples():
    print(review.star_rating, review.label, round(review.score, 2))
    print(review.review_body[:300], "\n")

1 POSITIVE 0.99
Don't get me wrong, the game concept is awesome. Graphics are great.  BUT  If you don't want to lose your progress when the Simcity Servers stop responding, don't buy this. There is no option to save locally just saved to their server so the smallest error will lose your work.  Overall it's a good g 

1 POSITIVE 1.0
Enjoys game 

1 POSITIVE 0.73
I wasted $7. im sure its a good game but i cant even get to the game part. i tried. a lot. 

1 POSITIVE 1.0
Dose not work right on PSVITA so was not happy but it is a fun game love twisted metal games. 

2 POSITIVE 0.87
I bought this game for my 8 year old daughter who likes the kingdom of keflings game on xbox. The graphics are very simple and it is a basic strategy/building style game. Unfortunately, it looks like it would have been cutting edge 15 to 20 years ago, but it really didn't entertain my kids for more  



**For each of the five, why do you think the model got it wrong? Was it the model, or was the customer's rating misleading?**

1. **1 star / POSITIVE (0.99)**: The SimCity review praises the concept and graphics, then complains that the servers lose your progress, and even ends with "Overall it's a good g[ame]". The text is genuinely mixed and leans positive, but the customer gave 1 star because of one dealbreaker. **Mostly the rating**: it reflects the server problem, not the overall tone of what they wrote.
2. **1 star / POSITIVE (1.0)**: "Enjoys game." That's clearly positive, and the model read it correctly. **The customer's rating is misleading**, maybe a misclick, or they were rating something other than the game.
3. **1 star / POSITIVE (0.73)**: "I wasted $7... im sure its a good game but i cant even get to the game part." This is clearly negative, but the model latched onto "good game" and missed "wasted" and "cant even". **Model's fault**, though its lower confidence (0.73) shows it was unsure.
4. **1 star / POSITIVE (1.0)**: "Dose not work right on PSVITA so was not happy but it is a fun game love twisted metal games." The text ends on praise for the game, while the 1 star is about it not working on their device. **Both**: the review is mixed, and the rating punishes a technical problem rather than the game.
5. **2 stars / POSITIVE (0.87)**: The Keflings review is a disappointed parent ("Unfortunately", "really didn't entertain my kids"), but it uses positive-sounding words like "likes" and "cutting edge" in a negative context. **Model's fault**: it read individual words instead of the overall meaning.

Overall, about **2 of 5** were the model's fault (#3 and #5), **1** was clearly a misleading rating (#2), and **2** were genuinely mixed reviews where the text praises the game but the rating punishes a technical problem (#1 and #4). Also worth noting: the model gave 1.0 confidence on two of these mistakes, so a high score doesn't mean it's right.

## Part 4: A Model That Predicts Stars

### Step 13: Predict the stars

In [13]:
star_model = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
)
results = star_model(sample["review_body"].tolist(), truncation=True)
sample["pred_stars"] = [int(r["label"][0]) for r in results]  # "4 stars" -> 4

print("Exact:", (sample["pred_stars"] == sample["star_rating"]).mean())
off_by = (sample["pred_stars"] - sample["star_rating"]).abs()
print("Within one star:", (off_by <= 1).mean())

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Exact: 0.574
Within one star: 0.912


**How often does it get the exact star rating? How often within one star?**
Exact: **57.4%**. Within one star: **91.2%**. Exact looks low, but random guessing on 5 balanced classes would only be 20%, so it's almost 3x better than chance. And being within one star 91% of the time means that when it misses, it's usually a near miss (calling a 4 a 5, or a 1 a 2), not calling a 1-star review a 5.

**Your team is building a sentiment dashboard for a game publisher. Which of the two models would you use, and why?**
I'd use the nlptown star model. It was trained on product reviews (closer to our data than DistilBERT's movie review sentences), it can show neutral/mixed sentiment instead of forcing a yes/no, and its 1 to 5 output maps directly to the star ratings a publisher already tracks. That means you can trend average predicted stars over time and flag products where the predicted and actual ratings disagree. DistilBERT hit 87.75% on a simple positive/negative split, but it threw away the 3-star reviews entirely and leaned negative on 4 and 5-star ones. For a dashboard, nuance matters more than a binary call, and

## Part 5: Zero-Shot: What Are People Complaining About?

### Step 14: Classify 20 one-star reviews

In [14]:
zsc = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

labels = ["price", "bugs or crashes", "account or login", "gameplay", "download or install"]
results = zsc(complaints["review_body"].tolist(), candidate_labels=labels)
complaints["topic"] = [r["labels"][0] for r in results]
complaints["topic"].value_counts()

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

topic
download or install    5
bugs or crashes        5
account or login       4
price                  3
gameplay               3
Name: count, dtype: int64

**What do customers complain about most?**
The most common topics were a tie between **download or install** (**5** of 20) and **bugs or crashes** (**5** of 20), followed by **account or login** (**4**), then **price** and **gameplay** (**3** each). Together, the technical categories (download/install, bugs/crashes, account/login) make up **14 of 20** complaints, while only **3** are about gameplay itself. That fits what we saw in Step 5: many of these "game" reviews are really about getting a code, an account or a download to work, not about the game. With only 20 reviews, though, a difference of one or two isn't meaningful.

In [15]:
for review in complaints.head(3).itertuples():
    print("TOPIC:", review.topic)
    print(review.review_body[:300], "\n")

TOPIC: price
Terrible, confusing interface. Looks like a cartoon. Didn't really like the long, drawn out stories in my player. Game play and graphics weren't that much improved $30 extra we had to pay (Now we get VC yea! not). 

TOPIC: download or install
I too installed and tried to play Cobi Treasure Island. My PC does meet the proper requirements so I do not know what the problem is. 

TOPIC: gameplay
If you liked \\"Heavy Rain\\", you'll like this.  However, if you'd like for your games to have a faster pace than 'the English Patient', you'll want to skip this title in favor of something more exciting, like \\"Parking Simulator 4\\". 



**Does the model's choice make sense?**
Review 1 was tagged **price**, which is only partly right. It does mention paying "$30 extra" and the virtual currency, but most of the review is about the interface, the cartoony look, the long stories and gameplay/graphics that "weren't that much improved", which is really **gameplay**. The model seems to have latched onto the dollar amount. Review 2 was tagged **download or install**, and that fits: they installed the game, their PC meets the requirements, and it still won't work. (You could argue for "bugs or crashes", but install fits well.) Review 3 was tagged **gameplay**, and that's a good call. The complaint is about slow pacing, made through sarcastic comparisons ("the English Patient", "Parking Simulator 4"), and the model got the right category even without obvious keywords. Overall it gets the gist when there's one clear complaint, but it struggles when a review mentions several problems, because it can only pick one top label, and a concrete detail like a price can pull it away from the main point.

### Step 15: Change the labels

In [16]:
more_labels = labels + ["technical problems", "HR policy"]
new_results = zsc(complaints["review_body"].head(5).tolist(), candidate_labels=more_labels)
for old, new in zip(results[:5], new_results):
    print("Before:", old["labels"][0], round(old["scores"][0], 2))
    print("After: ", new["labels"][0], round(new["scores"][0], 2), "\n")

Before: price 0.66
After:  price 0.59 

Before: download or install 0.41
After:  download or install 0.3 

Before: gameplay 0.36
After:  gameplay 0.32 

Before: bugs or crashes 0.53
After:  technical problems 0.7 

Before: download or install 0.79
After:  download or install 0.45 



**What happened to the top labels and their scores?**
4 of the 5 reviews kept the same top label, but every one of those scores dropped: price went from 0.66 to 0.59, download or install from 0.41 to 0.30 and from 0.79 to 0.45, and gameplay from 0.36 to 0.32. One review flipped from **bugs or crashes (0.53)** to **technical problems (0.70)**, so the new overlapping label didn't just steal share, it won outright with a higher score. "HR policy" never won for any review, but it still took a small slice of each review's score.

**The scores for each review add up to 1. How does that explain what you saw?**
The scores are relative, not absolute. The model splits 100% of its confidence across whatever labels you give it. When "technical problems" overlaps "download or install" and "bugs or crashes", it soaks up probability that used to go to those labels. That's why the 0.79 download review dropped to 0.45 even though the review didn't change. And when the new label fits better than any of the old ones, it can take over completely, like the bugs review that flipped to technical problems at 0.70. Even an irrelevant label like "HR policy" grabs a little share, which pushes every other score down.

**What does this tell you about choosing labels for a real product?**
The labels are part of the model. They should be mutually exclusive and cover the real complaint types (MECE), with no overlapping synonyms or junk categories, otherwise the counts shift for reasons that have nothing to do with customers. Adding one broad label like "technical problems" could swallow several specific ones and make it look like complaints changed when only the labels did. You should test label sets on real reviews before trusting the counts, and only compare results that used the same label set.